# Clase 3: Importancia del preprocesamiento después del EDA

### Machine Learning (RIA-805) · Bloque 1 · Recomendación de cultivos

**Conexión con la clase anterior:** el EDA nos permitió observar la estructura y el comportamiento de los datos. Ahora prepararemos las variables y comprobaremos cómo evitar la fuga de información.

**Objetivos de aprendizaje:**
- Separar predictores y objetivo, y reservar un conjunto de prueba antes de ajustar transformaciones.
- Imputar valores faltantes y escalar características usando solo información de entrenamiento.
- Comparar un flujo correcto con un ejemplo intencional de *data leakage* y explicar por qué difieren.

**Ruta de la clase:** carga → faltantes → separación train/test → imputación → escalado → evaluación y demostración de leakage → funciones reutilizables.

## 1. Cargar el dataset Parquet

El archivo `data/processed/crop.parquet` contiene las observaciones revisadas en el análisis anterior. En la siguiente celda mostramos su tamaño, algunas filas y los tipos de datos para confirmar con qué vamos a trabajar.

Avanzaremos por cada paso y observaremos las tablas que resultan de cada transformación.


In [ ]:
"""Practica guiada: preprocesamiento y prevencion de data leakage."""

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler


def mostrar_tabla(titulo, tabla):
    """Muestra un titulo y una tabla de pandas durante la explicacion."""
    print(titulo)
    display(tabla)


# Buscar la raiz hace que el notebook funcione aunque Jupyter se abra desde otra carpeta.
project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "data" / "processed" / "crop.parquet").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "No se encontro data/processed/crop.parquet desde el directorio actual."
    )

# El parquet es la salida del EDA; mostramos su estructura antes de transformarlo.
source_path = project_root / "data" / "processed" / "crop.parquet"
df = pd.read_parquet(source_path)
print(f"Dataset cargado desde: {source_path}")
print(f"Filas y columnas: {df.shape}")
mostrar_tabla("Primeras filas del dataset:", df.head())
mostrar_tabla("Tipos de datos:", df.dtypes.rename("tipo").to_frame())

## 2. Simular valores faltantes

En datos reales pueden faltar mediciones. Para observar el efecto, introduciremos un 5% de valores `NaN` en `N`, `P`, `K` y `temperature`. Usamos una semilla fija para que el ejemplo sea reproducible, y seleccionamos exactamente el 5% de las filas en cada columna.

In [ ]:
# 2. Simular NaN con seleccion aleatoria reproducible (5% exacto por columna).
missing_value_columns = ["N", "P", "K", "temperature"]
target_column = "label"
required_columns = missing_value_columns + [target_column]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas en el parquet: {missing_columns}")

feature_columns = [column for column in df.columns if column != target_column]
df_simulado = df.copy()
rng = np.random.default_rng(42)  # Misma mascara de faltantes en cada ejecucion.
missing_count = int(round(0.05 * len(df_simulado)))
for column in missing_value_columns:
    # Elegir filas sin reemplazo asegura exactamente 5% en cada columna.
    missing_indices = rng.choice(df_simulado.index, size=missing_count, replace=False)
    df_simulado.loc[missing_indices, column] = np.nan

print(f"Se asignaron {missing_count} NaN por cada columna (5% de {len(df_simulado)} filas).")
filas_con_faltantes = df_simulado[
    df_simulado[missing_value_columns].isna().any(axis=1)
]
mostrar_tabla(
    "Filas de ejemplo que contienen valores faltantes:",
    filas_con_faltantes[feature_columns + [target_column]].head(12),
)

## 3. Contar los valores faltantes

Antes de imputar, contamos los `NaN` en cada predictor. En la celda siguiente usamos las variables de entrada `X` para ver cuantos valores faltan por columna.

In [ ]:
# 3. Crear X y revisar los valores faltantes antes de imputar.
X = df_simulado[feature_columns].copy()
missing_before = X.isna().sum().rename("NaN antes de imputar")
mostrar_tabla("Valores faltantes por predictor:", missing_before.to_frame())
mostrar_tabla("Primeras filas de X:", X.head())

## 4. Separar X e y

In [ ]:
# 4. Definir la variable objetivo y mostrar algunos ejemplos.
y = df_simulado[target_column].copy()
mostrar_tabla("Primeras etiquetas de y (label):", y.head().to_frame())
print(f"Numero de predictores en X: {X.shape[1]}")

## 5. Separar entrenamiento y prueba

In [ ]:
# 5. Separar antes de aprender medianas o escalas para que test no filtre informacion.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,  # Conserva la proporcion de cultivos en train y test.
    random_state=42,
)
print(f"Filas en train: {X_train.shape[0]} ({X_train.shape[0] / len(X):.0%})")
print(f"Filas en test:  {X_test.shape[0]} ({X_test.shape[0] / len(X):.0%})")
mostrar_tabla(
    "Distribucion de clases en train (proporcion):",
    y_train.value_counts(normalize=True).sort_index().rename("proporcion").to_frame(),
)
mostrar_tabla("Ejemplo de X_train (aun puede contener NaN):", X_train.head())
mostrar_tabla("Ejemplo de X_test (aun puede contener NaN):", X_test.head())

## 6. Imputar con la mediana

In [ ]:
# 6. fit aprende las medianas unicamente de train; transform aplica esas medianas a test.
imputer = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train), columns=feature_columns, index=X_train.index
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test), columns=feature_columns, index=X_test.index
)
X_imputed = pd.concat([X_train_imputed, X_test_imputed]).loc[X.index]

medianas = pd.Series(imputer.statistics_, index=feature_columns, name="mediana aprendida en train")
mostrar_tabla("Mediana calculada para cada predictor:", medianas.to_frame())
missing_after = X_imputed.isna().sum().rename("NaN despues de imputar")
mostrar_tabla(
    "Comparacion de faltantes antes y despues de imputar:",
    pd.concat([missing_before, missing_after], axis=1),
)
mostrar_tabla("Muestras despues de imputar:", X_imputed.head(10))

## 7. Escalar con RobustScaler sin leakage

`RobustScaler` centra cada variable con su mediana y la escala usando el rango intercuartil. En el flujo correcto, el scaler se ajusta con train y solo transforma test; por eso test no participa en el calculo de medianas ni rangos.

In [ ]:
# 7. fit_transform solo en train; en test se usa transform con la misma escala aprendida.
scaler_correcto = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler_correcto.fit_transform(X_train_imputed),
    columns=feature_columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler_correcto.transform(X_test_imputed),
    columns=feature_columns,
    index=X_test.index,
)

mostrar_tabla(
    "Medianas de train escalado (deben estar cerca de 0):",
    X_train_scaled.describe().loc[["mean", "50%", "min", "max"]],
)
mostrar_tabla("Muestras de train escalado:", X_train_scaled.head())
mostrar_tabla("Muestras de test escalado:", X_test_scaled.head())

## 8. Entrenar Random Forest y medir accuracy

Entrenamos el `RandomForestClassifier` con los datos escalados de train y medimos su accuracy sobre test. El conjunto de prueba se mantiene reservado hasta esta evaluacion.

In [ ]:
# 8. Entrenar solamente con train y reservar test para medir generalizacion.
modelo_correcto = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
modelo_correcto.fit(X_train_scaled, y_train)
predicciones_correctas = modelo_correcto.predict(X_test_scaled)
accuracy_correcto = accuracy_score(y_test, predicciones_correctas)
print(f"Accuracy con preprocesamiento correcto: {accuracy_correcto:.4f}")
mostrar_tabla(
    "Ejemplos de etiquetas reales y predichas:",
    pd.DataFrame(
        {"real": y_test.head(10), "prediccion": predicciones_correctas[:10]},
        index=y_test.head(10).index,
    ),
)

## 9. Repetir con data leakage intencional

En este flujo incorrecto ajustamos `RobustScaler` con todas las filas imputadas, incluidas las que pertenecen a test, antes de extraer las particiones para este escalado. El scaler conoce estadisticas del conjunto de prueba. Mantenemos los mismos indices y modelo para aislar el efecto de esa fuga.

In [ ]:
# 9. FUGA INTENCIONAL: ajustar el scaler con todas las filas revela tambien estadisticas de test.
scaler_con_fuga = RobustScaler()
X_all_scaled = pd.DataFrame(
    scaler_con_fuga.fit_transform(X_imputed),
    columns=feature_columns,
    index=X.index,
)
# Seleccionar los mismos indices permite comparar unicamente el efecto de la fuga.
X_train_leak = X_all_scaled.loc[X_train.index]
X_test_leak = X_all_scaled.loc[X_test.index]

mostrar_tabla("Muestras escaladas con fuga (train):", X_train_leak.head())
mostrar_tabla("Muestras escaladas con fuga (test):", X_test_leak.head())

modelo_con_fuga = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
modelo_con_fuga.fit(X_train_leak, y_train)
predicciones_con_fuga = modelo_con_fuga.predict(X_test_leak)
accuracy_con_fuga = accuracy_score(y_test, predicciones_con_fuga)
print(f"Accuracy escalando todo antes de separar (con fuga): {accuracy_con_fuga:.4f}")

## 10. Comparar las accuracies

Las dos mediciones usan el mismo conjunto de prueba. Puede que el resultado del bosque aleatorio sea igual en ambos casos: los arboles suelen depender del orden de los valores y no de su escala. Eso no vuelve correcto el leakage; otros modelos y datos pueden verse afectados.

In [ ]:
# 10. Comparar accuracy en las mismas filas de test; la escala puede no cambiar un arbol.
comparacion = pd.DataFrame(
    {
        "proceso": [
            "Correcto: scaler ajustado solo en train",
            "Incorrecto: scaler ajustado en train y test",
        ],
        "accuracy_test": [accuracy_correcto, accuracy_con_fuga],
    }
)
mostrar_tabla("Comparacion de accuracy:", comparacion)

## 11-12. Guardar train procesado y scaler

Guardamos solo el conjunto de entrenamiento, con predictores imputados y escalados por el `RobustScaler` correcto, junto con `label`. El scaler que se guarda es el ajustado unicamente con train; se necesitara para preparar futuros datos con las mismas transformaciones.

In [ ]:
# 11. Guardar unicamente train ya imputado y escalado, junto con la etiqueta.
processed_dir = project_root / "data" / "processed"
models_dir = project_root / "models"
processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

train_processed = X_train_scaled.copy()
train_processed[target_column] = y_train
train_path = processed_dir / "crop_train.parquet"
train_processed.to_parquet(train_path, index=False)

# 12. Persistir el scaler ajustado solo con train, nunca el scaler con fuga.
scaler_path = models_dir / "scaler.pkl"
joblib.dump(scaler_correcto, scaler_path)

print(f"Dataset de train guardado en: {train_path}")
print(f"Scaler guardado en: {scaler_path}")
print(f"Forma del dataset guardado: {train_processed.shape}")
mostrar_tabla("Vista de las filas guardadas:", train_processed.head())

## Funciones reutilizables

El ejemplo anterior muestra cada transformacion por separado para aprender el proceso. En un proyecto, estas operaciones suelen guardarse como funciones. La regla sigue siendo la misma: primero dividir; luego ajustar imputador y scaler solo con train, y usar `transform` para test.

Estas funciones devuelven tambien los objetos ajustados (`imputer` y `scaler`) para reutilizarlos al preparar datos nuevos.

In [ ]:
def load_raw_data(path: str | Path) -> pd.DataFrame:
    """Carga datos desde un archivo Parquet."""
    return pd.read_parquet(path)


def split_data(
    df: pd.DataFrame,
    target: str,
    test_size: float = 0.2,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Separa predictores y objetivo en train/test de forma estratificada."""
    if target not in df.columns:
        raise ValueError(f"No existe la columna objetivo {target!r}.")

    X = df.drop(columns=target)
    y = df[target]
    return train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )


def handle_missing(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    strategy: str = "median",
) -> tuple[pd.DataFrame, pd.DataFrame, SimpleImputer]:
    """Imputa train/test; aprende la estrategia solo desde train."""
    if not X_train.columns.equals(X_test.columns):
        raise ValueError("Train y test deben tener las mismas columnas y en el mismo orden.")

    imputer = SimpleImputer(strategy=strategy)
    train_clean = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    test_clean = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )
    return train_clean, test_clean, imputer


def scale_features(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    scaler_type: str = "robust",
) -> tuple[pd.DataFrame, pd.DataFrame, RobustScaler | StandardScaler]:
    """Ajusta el scaler en train y transforma train/test con la misma escala."""
    if not X_train.columns.equals(X_test.columns):
        raise ValueError("Train y test deben tener las mismas columnas y en el mismo orden.")

    scaler_classes = {"robust": RobustScaler, "standard": StandardScaler}
    if scaler_type not in scaler_classes:
        raise ValueError("scaler_type debe ser 'robust' o 'standard'.")

    scaler = scaler_classes[scaler_type]()
    train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )
    return train_scaled, test_scaled, scaler

In [ ]:
# Ejemplo de uso modular: dividir primero y ajustar cada transformador solo con train.
df_recargado = load_raw_data(source_path)
print(f"Dataset recargado por la funcion: {df_recargado.shape}")

X_train_demo, X_test_demo, y_train_demo, y_test_demo = split_data(
    df_simulado,
    target=target_column,
)
X_train_limpio, X_test_limpio, imputer_demo = handle_missing(
    X_train_demo,
    X_test_demo,
    strategy="median",
)
X_train_escalado, X_test_escalado, scaler_demo = scale_features(
    X_train_limpio,
    X_test_limpio,
    scaler_type="robust",
)

print(f"Train: {X_train_escalado.shape}; test: {X_test_escalado.shape}")
print(f"Filas de prueba reservadas para evaluacion: {len(y_test_demo)}")
mostrar_tabla("Resultado de las funciones (train):", X_train_escalado.head())
mostrar_tabla("Resultado de las funciones (test):", X_test_escalado.head())
print(f"Imputador guardado para reutilizar: {type(imputer_demo).__name__}")
print(f"Scaler guardado para reutilizar: {type(scaler_demo).__name__}")

## 13. Cierre

Revisa el conteo de faltantes, la particion, las medianas aprendidas, los datos escalados, las predicciones y los archivos que acabamos de guardar.

In [ ]:
# 13. Mensaje final solicitado.
print("Preprocesamiento completado. Revisa data/processed/ y models/")